In [1]:
import os

import numpy as np
import torch

In [2]:
k = "9"
d_model = 1024
device = "cuda:1"
layers = [13, 14, 15]
expansion_factor = 16
tokens_dir = os.path.expanduser("~/group-sae/feature_analysis/tokens/pythia-410m/{k}")
features_dir = os.path.expanduser("~/group-sae/feature_analysis/features/pythia-410m/{k}")
activations_dir = os.path.expanduser("~/group-sae/feature_analysis/activations/pythia-410m/{k}")

In [3]:
torch.cuda.set_device(device)

In [4]:
# Load tokens
tokens = torch.from_numpy((np.load(os.path.join(tokens_dir.format(k="9"), "all_tokens.npy"))))

In [5]:
group = {}
for layer in layers:
    features = torch.from_numpy(
        (np.load(os.path.join(features_dir.format(k="9"), f"blocks.{layer}.hook_resid_post.npy")))
    ).reshape(-1, 128)

    activations = torch.from_numpy(
        (np.load(os.path.join(activations_dir.format(k="9"), f"blocks.{layer}.hook_resid_post.npy")))
    ).reshape(-1, 128)

    group[layer] = {
        "features": features,
        "activations": activations,
    }

In [6]:
baseline = {}
for layer in layers:
    features = torch.from_numpy(
        (np.load(os.path.join(features_dir.format(k="baseline"), f"blocks.{layer}.hook_resid_post.npy")))
    ).reshape(-1, 128)

    activations = torch.from_numpy(
        (np.load(os.path.join(activations_dir.format(k="baseline"), f"blocks.{layer}.hook_resid_post.npy")))
    ).reshape(-1, 128)

    baseline[layer] = {
        "features": features,
        "activations": activations,
    }

In [7]:
from transformers import AutoTokenizer

# Load the tokenizer for pythia-410m
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-410m")


def get_top_activating_tokens(
    tokens: torch.Tensor,
    features: torch.Tensor,
    activations: torch.Tensor,
    feature_idx,
    top_k=5,
    context_length=16,
    print_highlighted_only: bool = True,
):
    """
    Find the top-k most activating tokens for a given feature and decode them with context.

    Args:
        tokens: Tensor of token IDs (reshaped to (-1,))
        features: Tensor of feature activations (reshaped to (-1, 128))
        activations: Tensor of activation values (reshaped to (-1, 128))
        feature_idx: The feature index to analyze
        top_k: Number of top activating tokens to return
        context_length: Number of tokens before and after to show as context
    """
    print(f"=== TOP {top_k} ACTIVATING TOKENS FOR FEATURE {feature_idx} ===")

    # Find all positions where this feature appears
    feature_positions = torch.where(features == feature_idx)

    if len(feature_positions[0]) == 0:
        print(f"Feature {feature_idx} never activates!")
        return

    # Get the activation values for this feature at all positions where it appears
    activation_values = activations[feature_positions]

    # Get top-k positions with highest activations
    top_k_indices = torch.topk(activation_values, min(top_k, len(activation_values))).indices

    print(f"Feature {feature_idx} activates {len(feature_positions[0])} times")
    print(f"Activation range: {activation_values.min():.4f} to {activation_values.max():.4f}")
    print()

    for rank, idx in enumerate(top_k_indices):
        token_idx = feature_positions[0][idx].item()
        activation_value = activation_values[idx].item()

        # Calculate the original token position in the sequence
        # Since we reshaped to (-1, 128), we need to find the original sequence position
        batch_size = tokens.shape[1] if len(tokens.shape) > 1 else len(tokens)
        original_token_pos = int(token_idx % batch_size)
        sequence_idx = int(token_idx // batch_size)

        # Get the context around this token
        start_pos = max(0, original_token_pos - context_length)
        end_pos = min(
            len(tokens) if len(tokens.shape) == 1 else tokens.shape[1],
            original_token_pos + context_length + 1,
        )

        if len(tokens.shape) == 1:
            # 1D tokens array
            context_tokens = tokens[start_pos:end_pos]
            target_token = tokens[original_token_pos]
        else:
            # 2D tokens array
            context_tokens = tokens[sequence_idx, start_pos:end_pos]
            target_token = tokens[sequence_idx, original_token_pos]

        if not print_highlighted_only:
            context_text = tokenizer.decode(
                context_tokens.cpu().numpy(), skip_special_tokens=False
            )
            target_text = tokenizer.decode([target_token.cpu().item()], skip_special_tokens=False)
            print(f"Rank {rank + 1}: Activation = {activation_value:.4f}")
            print(
                f"Token position: {original_token_pos} "
                f"(sequence {sequence_idx if len(tokens.shape) > 1 else 0})"
            )
            print(f"Target token: '{target_text}' (ID: {target_token.item()})")
            print(f"Context: {repr(context_text)}")

        # Highlight the target token in context
        target_pos_in_context = original_token_pos - start_pos
        context_tokens_list = context_tokens.cpu().numpy().tolist()
        decoded_context_tokens = [
            tokenizer.decode([t], skip_special_tokens=False) for t in context_tokens_list
        ]

        highlighted_context = ""
        for i, token_text in enumerate(decoded_context_tokens):
            if i == target_pos_in_context:
                highlighted_context += f">>>{token_text}<<<"
            else:
                highlighted_context += token_text

        print(f"Highlighted: {repr(highlighted_context)}")
        print()

/home/fbelotti/group-sae/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
def select_features_by_activation_percent(
    features,
    min_percent: float,
    max_percent: float,
    batch_size=256,
):
    """
    Select features with activation percentage between min_percent and max_percent.

    Args:
        min_percent: Minimum activation percentage (0-1)
        max_percent: Maximum activation percentage (0-1)
    """
    # Ensure min_percent and max_percent are within valid range
    if not (0 <= min_percent <= 1) or not (0 <= max_percent <= 1):
        raise ValueError("min_percent and max_percent must be between 0 and 1")
    if min_percent > max_percent:
        raise ValueError("min_percent cannot be greater than max_percent")

    # Compute feature distribution
    features_dist = torch.zeros(d_model * expansion_factor, device="cuda")
    for i in range(0, features.shape[0], batch_size):
        features_dist.scatter_add_(
            0,
            features[i : i + batch_size].to("cuda").view(-1).long(),
            torch.ones(features[i : i + batch_size].numel(), device="cuda"),
        )

    # Get activation counts for all features
    features_dist_cpu = features_dist.cpu().numpy()

    # Calculate activation percentages
    activation_percentages = features_dist_cpu / features_dist.shape[0]

    # Select features within the specified range
    mask = (activation_percentages >= min_percent) & (activation_percentages <= max_percent)
    selected_indices = np.where(mask)[0]

    return features_dist, selected_indices

In [9]:
dist, indices = select_features_by_activation_percent(group[14]["features"], 0.8, 1.0)

In [10]:
indices

array([    1,    62,   152,   159,   166,   260,   286,   320,   342,
         349,   354,   361,   369,   377,   392,   453,   456,   487,
         495,   557,   613,   620,   658,   679,   718,   736,   768,
         775,   779,   780,   781,   786,   816,   820,   842,   853,
         880,   895,   897,   898,   914,   943,  1164,  1175,  1194,
        1196,  1245,  1266,  1291,  1299,  1329,  1348,  1350,  1369,
        1372,  1390,  1394,  1424,  1450,  1515,  1570,  1623,  1631,
        1633,  1653,  1683,  1733,  1742,  1804,  1820,  1821,  1852,
        1881,  1930,  1931,  1933,  1959,  2017,  2083,  2098,  2101,
        2107,  2153,  2193,  2208,  2213,  2217,  2261,  2265,  2288,
        2297,  2362,  2392,  2405,  2421,  2584,  2587,  2615,  2664,
        2668,  2707,  2747,  2756,  2760,  2790,  2806,  2822,  2828,
        2846,  2851,  2958,  2966,  2998,  3050,  3067,  3106,  3136,
        3138,  3161,  3173,  3193,  3211,  3236,  3242,  3246,  3249,
        3257,  3310,

In [11]:
"""Feature 225 is highly selective for the period token '.' (ID: 15) when
it appears immediately before function calls like Fatalf, Error, or Errorf in Go-like error handling code.
This feature likely detects the syntactic pattern of method or function invocation following a period, especially in error reporting statements.
"""

"""Feature 1024 is highly selective for the token ' 1' (ID: 337), especially when it appears in mathematical or sorting contexts.
The top activations occur when ' 1' is part of lists or sequences of numbers, often in instructions to sort or order numbers.
This feature likely detects the presence of the number 1 within numeric lists or sorting tasks.
"""

"""Feature 0 detects numeric values (especially 2-4 digit numbers) in scientific and technical contexts.
The top activations occur on tokens like ' 100', ' 170', ' 95', '4000', and ' 40' when they appear
with units of measurement (nM, °C, K, ms, GeV) or in scientific notation contexts.
This feature likely captures the pattern of numerical quantities with associated units in academic/scientific text."""

get_top_activating_tokens(
    tokens,
    group[15]["features"],
    group[15]["activations"],
    62,
    top_k=10,
    context_length=16,
)

=== TOP 10 ACTIVATING TOKENS FOR FEATURE 62 ===
Feature 62 activates 8655 times
Activation range: 0.0071 to 0.0578

Highlighted: '\nThe former method prepares the form for editing the entity:\n@RequestMapping>>>(<<<value = "/{id}", params = "form", produces = "text/'

Highlighted: ' should finish by the\ntime they knock off on Saturday."\n\n[Illustration>>>:<<< "WE ARE DESPERATE MEN AND WELL ARMED"]\n\n'

Highlighted: ' errors and merges the entity into the JPA context:\n@RequestMapping>>>(<<<method = RequestMethod.PUT, produces = "text/html")\npublic'

Highlighted: 'import java.lang.annotation.RetentionPolicy;\n\n@Retention>>>(<<<RetentionPolicy.SOURCE)\n@IntDef({\n    Corner.'

Highlighted: ' first need the following lemma relating Gromov-Hausdorff distances to quasi>>>-<<<isometries.\n\n\\[GH-quasi\\] Suppose $d_{'

Highlighted: ');\n    }\n\n    @Override\n    public String toString(int>>> indent<<<, boolean prettyPrint) {\n      String indentStr = prettyPrint ? T'

Highlighted: ' 

### Concordance

In [23]:
def compute_jaccard_similarity_cuda(
    baseline_features,
    group_features,
    M,
    batch_size=2048,
    similarity_threshold=0.5,
    min_activation_count=1,
):
    """
    Compute Jaccard similarity matrix and identify baseline-only and group-only features using efficient set operations.

    Args:
        baseline_features: Baseline-SAE features tensor (N, K) on CPU
        group_features: Group-SAE features tensor (N, K) on CPU
        M: Number of features (d_model * expansion_factor)
        batch_size: Batch size for memory efficiency
        similarity_threshold: Minimum Jaccard similarity to consider features as matched
        min_activation_count: Minimum activations to consider a feature as active

    Returns:
        dict with jaccard_similarity, baseline_only_features, group_only_features, shared_features
    """
    # Move to device and ensure int type
    baseline_act = baseline_features.to(device).int()
    group_act = group_features.to(device).int()

    N_tokens = baseline_act.shape[0]
    print(f"Computing Jaccard similarity matrix for {N_tokens:,} tokens...")
    print(f"Feature space size: {M} x {M} = {M*M:,} entries")

    # First, compute feature counts to identify active features
    print("Computing feature counts...")
    count_baseline = torch.bincount(baseline_act.reshape(-1), minlength=M)
    count_group = torch.bincount(group_act.reshape(-1), minlength=M)

    # Convert to CPU for set operations
    count_baseline_cpu = count_baseline.cpu().numpy()
    count_group_cpu = count_group.cpu().numpy()

    # Create sets of active features using efficient numpy operations
    baseline_active_set = set(np.where(count_baseline_cpu >= min_activation_count)[0])
    group_active_set = set(np.where(count_group_cpu >= min_activation_count)[0])

    print(f"Active baseline features: {len(baseline_active_set)}")
    print(f"Active group features: {len(group_active_set)}")

    # Use set operations to find feature relationships
    # Features that exist in both SAEs (but may have different similarities)
    potentially_shared = baseline_active_set & group_active_set

    # Features that only exist in one SAE or the other
    baseline_only_candidates = baseline_active_set - group_active_set
    group_only_candidates = group_active_set - baseline_active_set

    print(f"Features present in both SAEs: {len(potentially_shared)}")
    print(f"Baseline-only candidates: {len(baseline_only_candidates)}")
    print(f"Group-only candidates: {len(group_only_candidates)}")

    # Allocate global histogram vector for the AND matrix (flattened)
    global_hist = torch.zeros(M * M, device=device, dtype=torch.int32)

    # Process tokens in batches to avoid huge memory allocations
    print(f"Processing in batches of {batch_size}...")
    for i in range(0, N_tokens, batch_size):
        if i % (batch_size * 10) == 0:
            print(f"  Batch {i//batch_size + 1}/{(N_tokens + batch_size - 1)//batch_size}")

        batch_end = min(i + batch_size, N_tokens)

        # Get a batch of tokens, shape: (B, K)
        baseline_batch = baseline_act[i:batch_end]
        group_batch = group_act[i:batch_end]

        # Compute the outer (Cartesian) product for each token in the batch
        batch_linear_idx = baseline_batch.unsqueeze(2) * M + group_batch.unsqueeze(1)
        batch_linear_idx = batch_linear_idx.reshape(-1)

        # Count co-occurrences in this batch
        batch_hist = torch.bincount(batch_linear_idx, minlength=M * M)
        global_hist += batch_hist.to(torch.int32)

        # Clean up batch memory
        del baseline_batch, group_batch, batch_linear_idx, batch_hist
        if i % (batch_size * 4) == 0:
            torch.cuda.empty_cache()

    # Reshape global histogram into the AND matrix of shape (M, M)
    and_matrix = global_hist.reshape(M, M).to(torch.int32)
    del global_hist
    torch.cuda.empty_cache()

    # OR matrix: OR(i,j) = count_baseline[i] + count_group[j] - and_matrix[i,j]
    or_matrix = count_baseline.view(M, 1) + count_group.view(1, M) - and_matrix

    # Compute Jaccard similarity matrix
    jaccard_similarity = and_matrix.float() / (or_matrix.float() + 1e-8)
    jaccard_cpu = jaccard_similarity.cpu().numpy()

    # Now refine the categorization based on actual Jaccard similarities
    print("Analyzing feature similarities...")

    # Convert active sets to sorted lists for indexing
    baseline_active_list = sorted(list(baseline_active_set))
    group_active_list = sorted(list(group_active_set))

    shared_features = []
    baseline_matched = set()
    group_matched = set()

    # For each baseline feature, find best group match
    for baseline_feat in baseline_active_list:
        if len(group_active_list) == 0:
            continue

        # Get similarities only for active group features
        similarities = jaccard_cpu[baseline_feat, group_active_list]
        best_idx = similarities.argmax()
        best_similarity = similarities[best_idx]
        best_group_feat = group_active_list[best_idx]

        if best_similarity >= similarity_threshold:
            shared_features.append((baseline_feat, best_group_feat, best_similarity))
            baseline_matched.add(baseline_feat)
            group_matched.add(best_group_feat)

    # Final categorization using set operations
    baseline_only = baseline_active_set - baseline_matched
    group_only = group_active_set - group_matched

    # Convert back to sorted lists
    baseline_only = sorted(list(baseline_only))
    group_only = sorted(list(group_only))

    print("Final results:")
    print(f"Shared features (high similarity): {len(shared_features)}")
    print(f"Baseline-only features: {len(baseline_only)}")
    print(f"Group-only features: {len(group_only)}")

    # Additional statistics
    total_active_baseline = len(baseline_active_set)
    total_active_group = len(group_active_set)

    print("\nCoverage statistics:")
    print(
        f"Baseline coverage: {len(baseline_matched)}/{total_active_baseline} "
        f"({100*len(baseline_matched)/total_active_baseline:.1f}%)"
    )
    print(
        f"Group coverage: {len(group_matched)}/{total_active_group} "
        f"({100*len(group_matched)/total_active_group:.1f}%)"
    )

    # Clean up
    del baseline_act, group_act, and_matrix, or_matrix, count_baseline, count_group
    torch.cuda.empty_cache()

    return {
        "jaccard_similarity": jaccard_similarity,
        "shared_features": shared_features,
        "baseline_only_features": baseline_only,
        "group_only_features": group_only,
        "baseline_feature_counts": count_baseline_cpu,
        "group_feature_counts": count_group_cpu,
        "baseline_active_set": baseline_active_set,
        "group_active_set": group_active_set,
        "potentially_shared": potentially_shared,
        "stats": {
            "total_baseline_active": total_active_baseline,
            "total_group_active": total_active_group,
            "shared_count": len(shared_features),
            "baseline_only_count": len(baseline_only),
            "group_only_count": len(group_only),
            "potentially_shared_count": len(potentially_shared),
            "baseline_coverage": (
                len(baseline_matched) / total_active_baseline if total_active_baseline > 0 else 0
            ),
            "group_coverage": (
                len(group_matched) / total_active_group if total_active_group > 0 else 0
            ),
        },
    }


def features_concordance(
    features_a,
    features_b,
    similarity_threshold=0.8,
    min_activation_count=1,
    batch_size=4096,
):
    """
    Compare Group-SAE features with Baseline-SAE features using efficient CUDA Jaccard similarity.

    Args:
        group_layer: Which layer of Group-SAE to analyze
        baseline_layers: List of baseline layers to compare against
        similarity_threshold: Minimum Jaccard similarity (recommend 0.1-0.5)
        min_activation_count: Minimum activations to consider a feature as active
    """
    M = d_model * expansion_factor
    jaccard_stats = compute_jaccard_similarity_cuda(
        features_a,
        features_b,
        M,
        batch_size=batch_size,
        similarity_threshold=similarity_threshold,
        min_activation_count=min_activation_count,
    )
    del jaccard_stats["jaccard_similarity"]
    torch.cuda.empty_cache()

    return jaccard_stats

In [14]:
group_vs_baseline_concordance = {}
for group_layer in group.keys():
    for baseline_layer in baseline.keys():
        print(f"Comparing Group-SAE layer {group_layer} with Baseline-SAE layer {baseline_layer}...")
        if group_layer not in group_vs_baseline_concordance:
            group_vs_baseline_concordance[group_layer] = {}
        group_vs_baseline_concordance[group_layer][baseline_layer] = features_concordance(
            features_a=group[group_layer]["features"],
            features_b=baseline[baseline_layer]["features"],
            similarity_threshold=0.4,
            min_activation_count=1,
            batch_size=4096,
        )

Comparing Group-SAE layer 13 with Baseline-SAE layer 13...
Computing features_a feature counts...
features_a active features: 16023
Computing features_b feature counts...
features_b active features: 16239
Computing Jaccard similarity matrix for 1,003,520 tokens...
Feature space size: 16384 x 16384 = 268,435,456 entries
Computing feature counts...
Active baseline features: 16023
Active group features: 16239
Features present in both SAEs: 15883
Baseline-only candidates: 140
Group-only candidates: 356
Processing in batches of 4096...
  Batch 1/245
  Batch 11/245
  Batch 21/245
  Batch 31/245
  Batch 41/245
  Batch 51/245
  Batch 61/245
  Batch 71/245
  Batch 81/245
  Batch 91/245
  Batch 101/245
  Batch 111/245
  Batch 121/245
  Batch 131/245
  Batch 141/245
  Batch 151/245
  Batch 161/245
  Batch 171/245
  Batch 181/245
  Batch 191/245
  Batch 201/245
  Batch 211/245
  Batch 221/245
  Batch 231/245
  Batch 241/245
Analyzing feature similarities...
Final results:
Shared features (high sim

In [19]:
for group_layer, baseline_results in group_vs_baseline_concordance.items():
    for baseline_layer, stats in baseline_results.items():
        stats["shared_features"].sort(key=lambda x: x[2], reverse=True)

In [23]:
group_vs_baseline_concordance[14][15]["shared_features"][-1100:-900]

[(9761, 14860, 0.48021236),
 (378, 16086, 0.48020983),
 (6598, 2884, 0.47992218),
 (15516, 13421, 0.479798),
 (6472, 6826, 0.47978228),
 (4127, 4387, 0.47949922),
 (3321, 2938, 0.47948626),
 (13796, 11002, 0.47945204),
 (2219, 1507, 0.47928995),
 (8576, 9248, 0.47906977),
 (12136, 12654, 0.47906804),
 (3337, 13153, 0.47899568),
 (5860, 2207, 0.47886473),
 (13574, 13324, 0.47878787),
 (3510, 13899, 0.47873688),
 (6375, 85, 0.47867692),
 (12433, 12716, 0.47867516),
 (12557, 4231, 0.47866905),
 (5472, 9630, 0.4785825),
 (11579, 3352, 0.4785655),
 (3868, 3745, 0.4784689),
 (2474, 12288, 0.47815493),
 (6579, 6837, 0.47806296),
 (9259, 5997, 0.47803992),
 (10883, 5065, 0.4779906),
 (5944, 13262, 0.4779545),
 (4152, 9118, 0.47791165),
 (6762, 12922, 0.4778839),
 (12369, 13167, 0.47783577),
 (4034, 81, 0.4777877),
 (4672, 4288, 0.47778156),
 (9889, 10953, 0.47761193),
 (5770, 6511, 0.47760865),
 (11284, 1285, 0.47748384),
 (4363, 14546, 0.4774696),
 (13255, 14518, 0.47650513),
 (5780, 13573, 0

In [24]:
get_top_activating_tokens(
    tokens,
    baseline[15]["features"],
    baseline[15]["activations"],
    3156,
    top_k=5,
    context_length=16,
)

=== TOP 5 ACTIVATING TOKENS FOR FEATURE 3156 ===
Feature 3156 activates 2654 times
Activation range: 0.0091 to 0.3777

Highlighted: ' Florida" is a bit of a misnomer; she was already there.>>> Most<<< hurricane cloud shields are at least 300 miles in diameter, but it\'s only the'

Highlighted: ' are affiliated with paying millions to candidates, who are just running for legacy purposes.>>> Most<<< candidates/politicians, are Masters of Talk. Getting very little done, to'

Highlighted: ' grew up in a loving and close family with his parents and his baby sister.>>> Most<<< of his extended family, aunts uncles and grandparents, live in the same'

Highlighted: ' had bigger ideas." "She\'s always wanted to see the world." "And>>> most<<< of all, she\'s wanted to live on a shore, on a coast "'

Highlighted: '\n\n----"I got a lot of really good ideas, problem is,>>> most<<< of them suck."\n- George Carlin\n\n2. Gojira'



In [25]:
get_top_activating_tokens(
    tokens,
    group[14]["features"],
    group[14]["activations"],
    12257,
    top_k=5,
    context_length=16,
)

=== TOP 5 ACTIVATING TOKENS FOR FEATURE 12257 ===
Feature 12257 activates 1971 times
Activation range: 0.0119 to 0.3970

Highlighted: ' had bigger ideas." "She\'s always wanted to see the world." "And>>> most<<< of all, she\'s wanted to live on a shore, on a coast "'

Highlighted: ' to, I wanted to see them, I wanted to feel them, and\n>>>most<<< of all I wanted to taste them.\nShe pulled me back into the kiss'

Highlighted: ' are affiliated with paying millions to candidates, who are just running for legacy purposes.>>> Most<<< candidates/politicians, are Masters of Talk. Getting very little done, to'

Highlighted: ' Florida" is a bit of a misnomer; she was already there.>>> Most<<< hurricane cloud shields are at least 300 miles in diameter, but it\'s only the'

Highlighted: ' God as a loving therapist Who is always there to listen, to understand, and>>> most<<< importantly, not to judge us. This verse reminds us that above all, the'

